# Merge Apify exports and remove duplicates

Original file name: `Merging and Dataset Creation.ipynb`

**Not runnable from this repository.** It reads raw tweet text, which is not redistributed (see `data/README.md`). The code is shown for the method; outputs that displayed tweet text have been cleared, all other outputs are as executed.

In [1]:
#libraries
import os
import pandas as pd

# Merge daily tweets dataset into a monthly format

In [2]:
daily_folder = "../data/raw/apify_daily_exports" #daily folder dir
output_folder = "../data/raw/monthly_merged" #monthly dir

In [3]:
# List the monthly folder
monthly_folder = sorted(os.listdir(daily_folder))

In [4]:
# Iterate over the list of monthly folder
for month_folder in monthly_folder:
    folder_path = os.path.join(daily_folder, month_folder)
    
    # Check if the path is a directory
    if not os.path.isdir(folder_path):
        continue  # Skip this iteration and move to the next month_folder
    
    # List the csv files in the monthly directory
    csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]
    sorted_csv_files = sorted(csv_files)
    
    # Initialize an empty list
    dfs = []
    
    # Load and append each csv files to the list
    for file in sorted_csv_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        dfs.append(df)
    
    # Concatenate all the month dataframes sorted by timestamp
    month_merged_df = pd.concat(dfs, ignore_index=True)
    month_merged_df = month_merged_df.sort_values(by="timestamp")
    
    # Save the merged month dataframe to the output directory
    output_dir = os.path.join(output_folder, f"merged_2021_twitter_data_{month_folder}.csv")
    month_merged_df.to_csv(output_dir, index=False)

# Merge Montly data into a large dataset

In [5]:
folder_path = "../data/raw/monthly_merged"

In [6]:
# List all the csv files in the folder
month_files = sorted([file for file in os.listdir(folder_path) if file.endswith('.csv')])

# Load each CSV file, drop the 'adjusted_timestamp' column and merge into one dataset
dfs = []
for file in month_files:
    tdf = pd.read_csv(os.path.join(folder_path, file))
    tdf = tdf.drop(columns='adjusted_timestamp', errors='ignore')  # Drop the column if it exists
    dfs.append(tdf)

# Concatenate the dataframes, sort by timestamp, and reset the index
mdf = pd.concat(dfs, ignore_index=True).sort_values(by="timestamp").reset_index(drop=True)

# index to start from 1
mdf.index = range(1, len(mdf) + 1)

# Display
mdf.head()

In [7]:
last_row = mdf.tail(1)
last_row

In [8]:
# Define the output dir and save the datasets
output_dir = "../data/raw"

# If there's an 'index' column, drop it
if 'index' in mdf.columns:
    mdf = mdf.drop(columns=['index'])
    
last_row = mdf.tail(1)
last_row

In [9]:
# Drop rows after index 70938
mdf = mdf.loc[:70938]  

# Save the dataset with the index
output_path = os.path.join(output_dir, "raw_twitter_data_2021.csv")
mdf.to_csv(output_path, index=True, index_label='index')
mdf

In [10]:
# Drop the 'fullname' and 'username' columns
fdf = mdf.drop(columns=['fullname', 'username'])

# Drop rows after index 70938
fdf = fdf.loc[:70938]
fdf

# Remove duplicate

In [11]:
# sum of duplicate tweets
fdf.text.duplicated().sum()

20645

In [15]:
# remove duplicate tweets
fdf.drop_duplicates(subset='text', inplace=True, keep='last')
fdf.shape

(50293, 4)

In [13]:
fdf.text.duplicated().sum()

0

# Final Dataset

In [14]:
# Save the dataset without the 'fullname' and 'username' columns
output_path = os.path.join(output_dir, "twitter_2021_dataset.csv")
fdf.to_csv(output_path, index=False) 
fdf